Assignment no 1.Design and implement a basic simulation of a Federated Learning system in Python where
multiple clients train local models on their own data without sharing raw datasets. A central
server aggregates the locally trained model parameters using averaging to form a global model,
demonstrating the core federated learning workflow.

**Imports**

In [1]:
import numpy as np


**Client class (local training only)**

In [2]:
class Client:
    def __init__(self, x, y, lr=0.01):
        self.x = x
        self.y = y
        self.lr = lr

    def local_train(self, w, b, epochs=5):
        w_local, b_local = w, b

        for _ in range(epochs):
            y_pred = w_local * self.x + b_local
            dw = np.mean((y_pred - self.y) * self.x)
            db = np.mean(y_pred - self.y)

            w_local -= self.lr * dw
            b_local -= self.lr * db

        return w_local, b_local


**Server (global coordination + FedAvg)**

In [3]:
class Server:
    def __init__(self):
        self.w = np.random.randn()
        self.b = np.random.randn()

    def aggregate(self, client_params):
        self.w = np.mean([p[0] for p in client_params])
        self.b = np.mean([p[1] for p in client_params])


**Create clients with different private datasets**

In [4]:
np.random.seed(42)

clients = []
for i in range(5):
    x = np.random.randn(100)
    y = 3 * x + 2 + np.random.randn(100) * 0.5  # private noisy data
    clients.append(Client(x, y))


**Federated Learning loop**

In [5]:
server = Server()
rounds = 10

for r in range(rounds):
    client_updates = []

    for client in clients:
        w_new, b_new = client.local_train(server.w, server.b)
        client_updates.append((w_new, b_new))

    server.aggregate(client_updates)

    print(f"Round {r+1}: w={server.w:.4f}, b={server.b:.4f}")


Round 1: w=1.4796, b=0.9799
Round 2: w=1.5556, b=1.0324
Round 3: w=1.6278, b=1.0822
Round 4: w=1.6962, b=1.1294
Round 5: w=1.7611, b=1.1742
Round 6: w=1.8226, b=1.2167
Round 7: w=1.8810, b=1.2570
Round 8: w=1.9363, b=1.2953
Round 9: w=1.9888, b=1.3316
Round 10: w=2.0386, b=1.3661
